# Visualize Data

This notebook visualizes collected AC data available from different AClib2 scenarios. Uncomment one scenario in the next cell but one and rerun whole notebook.

In [ ]:
import itertools
import os
import sys

import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
from scipy import stats 

sys.path.append("../")
from helper import load_data, data_source_release, dist_helper

sc_dict = data_source_release.get_sc_dict()
data_dir = data_source_release.get_data_dir()

%matplotlib inline
plt.style.use('seaborn-v0_8-deep')


In [ ]:
def get_y_values(scenario):
    runtimes, _, _ = load_data.get_data(scenario=scenario, data_dir=data_dir, sc_dict=sc_dict, 
                                retrieve=sc_dict[scenario]['use'])

    print(runtimes.shape)
    y_flat = runtimes.flatten().reshape([-1, 1])
    y_max_ = np.max(y_flat)
    y_min_ = 0
    y_values = (y_flat - y_min_)/y_max_
    return y_values


def plot_fit(scenario, dists, dist_names, save=False):
    y_values = get_y_values(scenario=scenario)
    # --- PDF plot ---
    fig, ax = plt.subplots(figsize=(8, 5))

    # empirical histogram (PDF)
    ax.hist(y_values, bins=100, density=True, alpha=0.5, color="gray", label="Empirical PDF")

    # grid for evaluation
    x = np.linspace(min(y_values), max(y_values), 500)

    # loop over distributions
    for dist in dists:
        params = dist.fit(y_values, floc=0)
        pdf = dist.pdf(x, *params)
        # plot PDF
        ax.plot(x, pdf, lw=2, label=dist_names.get(dist.name, dist.name))

    ax.set_xlabel("running times")
    ax.set_ylabel("PDF")
    ax.legend()
    plt.tight_layout()
    if save:
        plt.savefig(f"./plots_fit/{scenario}.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_log_fit(scenario, dists, dist_names, save=False):
    y_values = get_y_values(scenario=scenario)
    # --- log-transform runtimes ---
    y_log = np.log(y_values[y_values > 0])   # avoid log(0)

    # --- PDF plot ---
    fig, ax = plt.subplots(figsize=(8, 5))

    # empirical histogram (PDF)
    ax.hist(y_log, bins=100, density=True, alpha=0.5, color="gray", label="Empirical PDF")

    # grid for evaluation
    x = np.linspace(min(y_log), max(y_log), 500)

    # loop over distributions
    for dist in dists:
        try:
            params = dist.fit(y_log)
        except:
            params = dist.fit(y_values)
        pdf = dist.pdf(x, *params)
        # plot PDF
        ax.plot(x, pdf, lw=2, label=dist_names.get(dist.name, dist.name))

    ax.set_xlabel("log(running times)")
    ax.set_ylabel("PDF")
    ax.legend()
    plt.tight_layout()
    if save:
        plt.savefig(f"./plots_fit/log_{scenario}.png", dpi=300, bbox_inches="tight")
    plt.show()


In [ ]:
# list of candidate distributions
DISTS = [
    stats.lognorm,
    stats.invgauss,
    stats.expon,
    stats.weibull_min,
    stats.norm
]
DIST_NAMES = {
    "lognorm": "LOG",
    "invgauss": "INV",
    "expon": "EXP",
    "weibull_min": "WEI",
    "norm": "N"
}
# Choose a scenario

# SCENARIO="lpg-zeno"
# SCENARIO="saps-CVVAR"
# SCENARIO="spear_qcp"
# SCENARIO="spear_swgcp"
# SCENARIO="yalsat_qcp"
# SCENARIO="yalsat_swgcp"
SCENARIO="clasp_factoring"

plot_fit(SCENARIO, DISTS, DIST_NAMES, save=True)
plot_log_fit(SCENARIO, DISTS, DIST_NAMES, save=True)


In [ ]:
# THIS IS JUST A DRAFT
dists = [
    stats.lognorm,
    stats.invgauss,
    stats.expon,
    stats.weibull_min,
    stats.norm
]
# --- data: keep only finite, strictly positive values ---
y = np.asarray(y_values)


# log-spaced bins so the histogram matches a log x-axis nicely
bins = np.logspace(np.log10(y.min()), np.log10(y.max()), 100)

# distributions to compare
dists = [stats.lognorm, stats.invgauss, stats.expon, stats.weibull_min, stats.norm]

fig, ax = plt.subplots(figsize=(8, 5))

# empirical PDF (on original scale)
ax.hist(y, bins=bins, density=True, alpha=0.3, label="Empirical PDF")

# x-grid for PDFs
x = np.logspace(np.log10(y.min()), np.log10(y.max()), 500)

# fit & plot
for dist in dists:
    # positive-support dists: fix loc=0 (common for runtimes)
    if dist in (stats.lognorm, stats.invgauss, stats.expon, stats.weibull_min):
        params = dist.fit(y, floc=0)
    else:
        params = dist.fit(y)  # e.g., norm

    pdf = dist.pdf(x, *params)
    ax.plot(x, pdf, lw=2, label=dist.name)

ax.set_xscale("log")          # <- key: log x-axis for detail near 0
ax.set_xlabel("runtime")
ax.set_ylabel("density")
ax.legend()
plt.tight_layout()
plt.show()